# Neural Networks for Chaotic Dynamical Systems: The Hénon Map

Chaotic dynamical systems are deterministic — they follow exact mathematical rules — yet their long-term behaviour is practically unpredictable due to extreme sensitivity to initial conditions. The **Hénon map** is one of the most studied examples of a low-dimensional chaotic attractor. In this notebook we ask: *can a neural network learn the underlying dynamics of a chaotic system purely from data?*

We frame the problem as **supervised sequence prediction**: given the current state $(x_n, y_n)$, predict the next state $(x_{n+1}, y_{n+1})$. We generate training data by simulating many trajectories, train a fully-connected network with PyTorch, and then analyse both short-term accuracy and long-term attractor structure. Along the way we will see why chaos makes long-range prediction fundamentally hard — even for a perfect model.

## 1. Background: The Hénon Map

The Hénon map is a two-dimensional discrete-time dynamical system introduced by Michel Hénon in 1976. Starting from an initial condition $(x_0, y_0)$, each successive state is computed by:

$$x_{n+1} = 1 - a\,x_n^2 + y_n$$
$$y_{n+1} = b\,x_n$$

With the **classical parameters** $a = 1.4$ and $b = 0.3$ the system exhibits fully developed chaos: trajectories never repeat, but they remain confined to a fractal structure called the **Hénon attractor**.

### Why is this hard to predict?

The hallmark of chaos is **sensitive dependence on initial conditions**. Two trajectories that start a distance $\delta_0$ apart will — on average — diverge exponentially:

$$\delta_n \approx \delta_0\, e^{\lambda n}$$

where $\lambda > 0$ is the **Lyapunov exponent**. For the Hénon map $\lambda \approx 0.42$, meaning that errors roughly double every $\ln 2 / 0.42 \approx 1.6$ steps. This is why even a very accurate model will eventually produce trajectories that diverge from the true ones.

### What can a neural network learn?

Even though long-term prediction is impossible in principle, a neural network *can* learn the one-step mapping $(x_n, y_n) \mapsto (x_{n+1}, y_{n+1})$ very well. Moreover, when we iterate the learned model autonomously, the generated trajectories tend to stay on the attractor — capturing the **global statistical structure** of the chaos even when individual trajectories diverge.

## 2. Imports

## 3. Generating Training Data

A single long trajectory would over-represent parts of the attractor that happen to be visited more often. Instead, we simulate **$m = 1000$ trajectories of $n = 100$ steps** from different initial conditions spread across the basin of attraction.

Each consecutive pair of states becomes one training sample:

$$\text{input} = (x_n,\, y_n), \qquad \text{target} = (x_{n+1},\, y_{n+1})$$

This gives us $m \times (n-1) = 99{,}000$ input–output pairs — enough to cover the attractor densely.

### Visualising the Hénon Attractor

Plotting all generated points reveals the characteristic banana-shaped fractal structure of the attractor. Points cluster along thin curved filaments — the self-similar signature of chaos.

## 4. Building the Dataset

We reshape the arrays into flat lists of $(x_n, y_n)$ input pairs and $(x_{n+1}, y_{n+1})$ target pairs, then apply a standard 80/20 train/test split.

## 5. Model Definition

We use a **fully-connected regression network** with four layers:

| Layer | Units | Activation |
|-------|-------|------------|
| Input | 2 | — |
| Hidden 1 | 64 | ReLU |
| Hidden 2 | 32 | ReLU |
| Hidden 3 | 16 | ReLU |
| Output | 2 | **Linear** |

The output layer is **linear** (no activation) because we are doing regression — predicting continuous coordinates, not class probabilities. The loss function is Mean Squared Error:

$$\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} \left\| \hat{\mathbf{s}}_i - \mathbf{s}_i \right\|^2$$

where $\mathbf{s}_i = (x_{n+1}, y_{n+1})$ is the true next state and $\hat{\mathbf{s}}_i$ is the network's prediction.

## 6. Training with Early Stopping

We train with the **Adam** optimiser (adaptive learning rates) and stop early if the validation loss does not improve for `patience = 10` consecutive epochs. At each epoch we track:

- **Training loss / MAE** — computed on the training mini-batches
- **Validation loss / MAE** — computed on the held-out validation split

When early stopping triggers, we restore the weights from the epoch with the best validation loss — this prevents overfitting.

## 7. Learning Curves

Plotting loss and MAE on a **logarithmic scale** makes it easy to see whether training has converged. Key things to look for:

- **Both curves descending together** → healthy training, no overfitting.
- **Validation curve rising while training curve falls** → overfitting; early stopping should have caught this.
- **Both curves plateauing** → the model has reached its capacity limit.

## 8. Autoregressive Trajectory Prediction

Once trained, we use the network **autoregressively**: we feed it a single initial condition and iteratively feed its own output back as input to generate a full trajectory.

$$\hat{\mathbf{s}}_{n+1} = f_{\theta}(\hat{\mathbf{s}}_n), \qquad \hat{\mathbf{s}}_0 = (x_0, y_0)$$

Any small one-step error gets amplified by the Lyapunov exponent over successive steps — this is why short-term prediction is accurate but long-term prediction inevitably diverges, even for a well-trained model. This is a fundamental property of chaos, not a failure of the network.

## 9. Short-term vs Long-term Accuracy

We compare the first and last 20 steps of the predicted and true trajectories. The network should track the true trajectory closely at the start, with errors growing over time due to the positive Lyapunov exponent.

## 10. Attractor Reconstruction

Although individual trajectories diverge, the network has internalised the *geometry* of the attractor. By iterating the model from a single starting point for many steps we can reconstruct the attractor shape — and compare it to the true one.

A good model should reproduce the characteristic banana shape and fractal filaments, even if no single predicted point matches the corresponding true point.

## 11. Summary and Key Takeaways

| | |
|---|---|
| **Task** | One-step regression: $(x_n, y_n) \to (x_{n+1}, y_{n+1})$ |
| **Architecture** | 4-layer MLP, ReLU activations, linear output |
| **Training** | Adam + MSE loss, early stopping on validation loss |
| **Short-term prediction** | Accurate — the network has learned the local dynamics |
| **Long-term prediction** | Diverges — unavoidable due to chaos ($\lambda > 0$) |
| **Attractor structure** | Recovered — the network has learned the global geometry |

### Why does the long-term prediction fail?

This is not a failure of the network — it is a mathematical impossibility. The Lyapunov exponent $\lambda \approx 0.42$ means that even a model error of $10^{-6}$ will grow to $\mathcal{O}(1)$ after roughly $6 \ln 10 / 0.42 \approx 33$ steps. No finite-precision model can overcome this.

### What has the network actually learned?

The network has learned the **invariant measure** of the attractor: the probability distribution over states that the Hénon map visits in the long run. This is why iterated predictions stay on the attractor even when they diverge from the true trajectory — the model has captured the *statistics* of chaos, not the individual trajectory.